<!-- DATA IO Exercise (not Complete)--> 

In [ ]:
from pyspark import SparkContext

sc = SparkContext("local[*]", "DataIO")

# Load the CSV file
lines = sc.textFile("data/sales_data.csv")

# Skip header line
header = lines.first()
data = lines.filter(lambda line: line != header)

print(f"Header: {header}")
print(f"Data records: {data.count()}")
print(f"First record: {data.first()}")

def parse_record(line):
    """Parse CSV line into structured data."""
    parts = line.split(",")
    return {
        "product_id": parts[0],
        "name": parts[1],
        "category": parts[2],
        "price": float(parts[3]),
        "quantity": int(parts[4])
    }

# Parse all records
parsed = data.map(parse_record)

# Show parsed data
for record in parsed.take(3):
    print(record)

# Calculate revenue for each product
revenue = parsed.map(lambda r: f"{r['product_id']},{r['name']},{r['price'] * r['quantity']:.2f}")

# Save to output directory
# YOUR CODE: Use saveAsTextFile to save revenue data

revenue.saveAsTextFile("output/revenue_data")


### Task 4: Load Multiple Files

# I don't know what this is all about????
# ======================================================================================


# YOUR CODE: Create sales_data_2.csv with more records
# YOUR CODE: Load all CSV files using wildcard pattern
# all_data = sc.textFile("sales_data*.csv")


### Task 5: Coalesce Output


# YOUR CODE: Use coalesce(1) before saveAsTextFile
# This creates a single output file instead of multiple parts


## Deliverables

# 1. `data_io.py` - Complete script
# 2. `sales_data.csv` - Input file
# 3. Output directory with processed results



## Definition of Done

# - [ ] Data loaded successfully from CSV
# - [ ] Header line filtered out
# - [ ] Records parsed into structured format
# - [ ] Revenue calculated and saved
# - [ ] Coalesce produces single output file
# - [ ] Script runs without errors

# ---

# ## Additional Resources
# - Written Content: `data-loading-and-saving-through-rdds.md`


<!-- RDD Actions -->

In [2]:
from pyspark import SparkContext

sc = SparkContext("local[*]", "RDDActions")

nums = sc.parallelize([10, 5, 8, 3, 15, 12, 7, 20, 1, 9])

all_nums = nums.collect()
print(f"All numbers: {all_nums}")

count = nums.count()
print(f"Count of numbers: {count}")

first = nums.first()
print(f"First number: {first}")

first3 = nums.take(3) # take returns x amount

top3 = nums.top(3) # top returns the largest x amount
print(f"Top 3 numbers: {top3}")

bot3 = nums.takeOrdered(3) # takeOrdered returns the smallest x amount
print(f"Bottom 3 numbers: {bot3}")



total = nums.reduce(lambda x, y: x + y)
print(f"Total sum of numbers: {total}")

maximum = nums.reduce(lambda x, y: max(x, y))
print(f"Maximum number: {maximum}")

minimum = nums.reduce(lambda x, y: min(x, y))
print(f"Minimum number: {minimum}")

folded_sum = 0 #idk what this is 
print(f"Folded sum of numbers: {folded_sum}")

colors = sc.parallelize(["red", "blue", "red", "green", "blue", "red", "yellow"])
color_counts = colors.countByValue()
print(f"Color counts: {color_counts}")

sc.stop()



All numbers: [10, 5, 8, 3, 15, 12, 7, 20, 1, 9]


Count of numbers: 10
First number: 10


Top 3 numbers: [20, 15, 12]
Bottom 3 numbers: [1, 3, 5]
Total sum of numbers: 90
Maximum number: 20
Minimum number: 1
Folded sum of numbers: 0
Color counts: defaultdict(<class 'int'>, {'red': 3, 'blue': 2, 'green': 1, 'yellow': 1})


<!-- RDD Basics -->

In [ ]:
from pyspark import SparkContext,SparkConf
conf = SparkConf().setMaster('local[*]').setAppName('RDD Basics')
sc = SparkContext.getOrCreate(conf)

numbers = sc.parallelize([1,2,3,4,5,6,7,8,9,10])

print(f"Numbers: {numbers.collect()}")
print(f"Partitions: {numbers.getNumPartitions()}")

print("============================")
numbers_2_the_sequel = sc.parallelize([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], 4)
print(f"Numbers: {numbers_2_the_sequel.collect()}")
print(f"Partitions: {numbers_2_the_sequel.getNumPartitions()}")

squared = numbers.map(lambda x: x ** 2)
print(f"Squares: {squared.collect()}")


prefixed = numbers.map(lambda x: f"num_{str(x)}")
print(f"Prefixed: {prefixed.collect()}")

evens = numbers.filter(lambda x: x % 2 == 0)
print(f"evens: {evens.collect()}")

greater_than_5 = numbers.filter(lambda x: x > 5)
print(f"big: {greater_than_5.collect()}")

combo = evens.intersection(greater_than_5)
print(f"combo: {combo.collect()}")

sentences = sc.parallelize([
    "Hello World",
    "Apache Spark is Fast",
    "PySpark is Python plus Spark"
])

words = sentences.flatMap(lambda x: x.split())
print(f"words: {words.collect()}")

# is there a way to just use flat map
word_lengths = words.map(lambda x: (x, len(x)))
print(f"words: {word_lengths.collect()}")


logs = sc.parallelize([
    "INFO: User logged in",
    "ERROR: Connection failed",
    "INFO: Data processed",
    "ERROR: Timeout occurred",
    "DEBUG: Cache hit"
])

# fix this

error_words = logs.filter(lambda x : x == "ERROR").flatMap(lambda x : x.split()).map(lambda x: x.upper())
print(f"Error Words: {error_words.collect()}")







Numbers: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Partitions: 20
Numbers: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Partitions: 4
Squares: [1, 4, 9, 16, 25, 36, 49, 64, 81, 100]
Prefixed: ['num_1', 'num_2', 'num_3', 'num_4', 'num_5', 'num_6', 'num_7', 'num_8', 'num_9', 'num_10']
evens: [2, 4, 6, 8, 10]
big: [6, 7, 8, 9, 10]


combo: [6, 8, 10]
words: ['Hello', 'World', 'Apache', 'Spark', 'is', 'Fast', 'PySpark', 'is', 'Python', 'plus', 'Spark']
words: [('Hello', 5), ('World', 5), ('Apache', 6), ('Spark', 5), ('is', 2), ('Fast', 4), ('PySpark', 7), ('is', 2), ('Python', 6), ('plus', 4), ('Spark', 5)]
Error Words: []


26/06/30 18:21:03 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 3020153 ms exceeds timeout 120000 ms
26/06/30 18:21:03 WARN SparkContext: Killing executors is not supported by current scheduler.
26/06/30 18:21:05 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$

<!-- Exercise_transformations (to be done later) -->